# 05 - CSV Exports for Tableau and Looker Studio
Exports 6 CSVs from PostgreSQL views.

| File | Destination | Dashboard |
|------|-------------|-----------|
| rfm_segments.csv | data/tableau/ | Tableau Executive |
| cohort_retention.csv | data/tableau/ | Tableau Executive |
| revenue_by_category.csv | data/tableau/ | Tableau Executive |
| category_sla.csv | data/tableau/ | Tableau Executive |
| seller_performance.csv | data/looker/ | Looker Operational |
| cancellation_analysis.csv | data/looker/ | Looker Operational |

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from dotenv import load_dotenv
from urllib.parse import quote_plus
import os

load_dotenv()
DB_PASS = os.getenv("DB_PASSWORD")
DB_USER = os.getenv("DB_USER", "postgres")
DB_HOST = os.getenv("DB_HOST", "localhost")
DB_PORT = os.getenv("DB_PORT", "5432")
DB_NAME = os.getenv("DB_NAME", "olist_ecommerce")

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{quote_plus(DB_PASS)}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

TABLEAU_PATH = r"D:\DataAnalyticsProjects\ecommerce-retail-analysis\data\tableau"
LOOKER_PATH  = r"D:\DataAnalyticsProjects\ecommerce-retail-analysis\data\looker"

os.makedirs(TABLEAU_PATH, exist_ok=True)
os.makedirs(LOOKER_PATH, exist_ok=True)

with engine.connect() as conn:
    from sqlalchemy import text
    result = conn.execute(text("SELECT 1"))
    print("✅ Connected.")

✅ Connected.


In [2]:
# All 6 CSV exports

exports = {
    "rfm_segments": {
        "query": "SELECT * FROM olist.v_rfm_scores",
        "path": TABLEAU_PATH,
        "filename": "rfm_segments.csv"
    },
    "cohort_retention": {
        "query": """
            SELECT * FROM olist.v_cohort_retention
            WHERE cohort_month != '2016-12-01'
            ORDER BY cohort_month, period_number
        """,
        "path": TABLEAU_PATH,
        "filename": "cohort_retention.csv"
    },
    "revenue_by_category": {
        "query": "SELECT * FROM olist.v_revenue_by_category ORDER BY order_month, product_category_name",
        "path": TABLEAU_PATH,
        "filename": "revenue_by_category.csv"
    },
    "category_sla": {
        "query": "SELECT * FROM olist.v_category_sla ORDER BY late_rate_pct DESC",
        "path": TABLEAU_PATH,
        "filename": "category_sla.csv"
    },
    "seller_performance": {
        "query": "SELECT * FROM olist.v_seller_performance ORDER BY late_rate_pct DESC",
        "path": LOOKER_PATH,
        "filename": "seller_performance.csv"
    },
    "cancellation_analysis": {
        "query": "SELECT * FROM olist.v_cancellation_analysis ORDER BY analysis_dimension, cancel_rate_pct DESC",
        "path": LOOKER_PATH,
        "filename": "cancellation_analysis.csv"
    }
}

print("Exporting CSVs...\n")
results = []

for name, config in exports.items():
    df = pd.read_sql(config["query"], engine)
    filepath = os.path.join(config["path"], config["filename"])
    df.to_csv(filepath, index=False)
    results.append({
        "file": config["filename"],
        "rows": len(df),
        "cols": len(df.columns),
        "destination": "tableau" if config["path"] == TABLEAU_PATH else "looker"
    })
    print(f"✅ {config['filename']:<35} {len(df):>7,} rows  {len(df.columns):>3} cols")

print(f"\nTotal files exported: {len(results)}")

Exporting CSVs...

✅ rfm_segments.csv                     93,470 rows   12 cols
✅ cohort_retention.csv                    217 rows    5 cols
✅ revenue_by_category.csv               1,252 rows   11 cols
✅ category_sla.csv                         62 rows   10 cols
✅ seller_performance.csv                1,238 rows   13 cols
✅ cancellation_analysis.csv               109 rows    6 cols

Total files exported: 6


In [3]:
# Validate all 6 files exist on disk with correct sizes

import glob

print("--- Tableau exports ---")
for f in sorted(glob.glob(os.path.join(TABLEAU_PATH, "*.csv"))):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<35} {size_kb:>8.1f} KB")

print("\n--- Looker exports ---")
for f in sorted(glob.glob(os.path.join(LOOKER_PATH, "*.csv"))):
    size_kb = os.path.getsize(f) / 1024
    print(f"  {os.path.basename(f):<35} {size_kb:>8.1f} KB")

--- Tableau exports ---
  category_sla.csv                         4.3 KB
  cohort_retention.csv                     6.2 KB
  revenue_by_category.csv                105.2 KB
  rfm_segments.csv                      8432.0 KB

--- Looker exports ---
  cancellation_analysis.csv                3.8 KB
  seller_performance.csv                 123.4 KB
